In [2]:
import pandas as pd
import os

data_path = "../Data/"

datasets = {
    "hospital_general_info":   pd.read_csv(data_path + "hospital_general_info.csv"),
    "readmissions_and_deaths": pd.read_csv(data_path + "readmissions_and_deaths.csv"),
    "medicare_spending":       pd.read_csv(data_path + "medicare_spending.csv"),
    "complications_and_hais":  pd.read_csv(data_path + "complications_and_hais.csv"),
    "healthcare_infections":   pd.read_csv(data_path + "healthcare_infections.csv"),
    "hcahps_patient_survey":   pd.read_csv(data_path + "hcahps_patient_survey.csv"),
    "unplanned_visits":        pd.read_csv(data_path + "unplanned_visits.csv"),
    "timely_effective_care":   pd.read_csv(data_path + "timely_effective_care.csv"),
    "payment_and_value":       pd.read_csv(data_path + "payment_and_value.csv")
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

C:\Users\Fatima zahra\AppData\Local\Temp\ipykernel_18272\2188171443.py:12: DtypeWarning: Columns (12,14,17,19) have mixed types. Specify dtype option on import or set low_memory=False.
  "hcahps_patient_survey":   pd.read_csv(data_path + "hcahps_patient_survey.csv"),
C:\Users\Fatima zahra\AppData\Local\Temp\ipykernel_18272\2188171443.py:13: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  "unplanned_visits":        pd.read_csv(data_path + "unplanned_visits.csv"),
C:\Users\Fatima zahra\AppData\Local\Temp\ipykernel_18272\2188171443.py:14: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  "timely_effective_care":   pd.read_csv(data_path + "timely_effective_care.csv"),


hospital_general_info: 5426 rows, 38 columns
readmissions_and_deaths: 95780 rows, 18 columns
medicare_spending: 63646 rows, 13 columns
complications_and_hais: 6943 rows, 12 columns
healthcare_infections: 172404 rows, 15 columns
hcahps_patient_survey: 325652 rows, 22 columns
unplanned_visits: 67046 rows, 20 columns
timely_effective_care: 138129 rows, 16 columns
payment_and_value: 18372 rows, 22 columns


In [3]:
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f" {name.upper()}")
    print(f"{'='*50}")
    null_pct = (df.isnull().sum() / len(df) * 100).round(1)
    null_pct = null_pct[null_pct > 0]
    if null_pct.empty:
        print("no nulls")
    else:
        print(null_pct.to_string())


 HOSPITAL_GENERAL_INFO
meets_criteria_for_birthing_friendly_designation    58.3
hospital_overall_rating_footnote                    52.6
mort_group_footnote                                 67.1
safety_group_footnote                               61.7
readm_group_footnote                                78.6
pt_exp_group_footnote                               58.1
te_group_footnote                                   82.7

 READMISSIONS_AND_DEATHS
footnote    53.2

 MEDICARE_SPENDING
no nulls

 COMPLICATIONS_AND_HAIS
address_line_2      100.0
countyparish          1.2
telephone_number      1.6
ownership_type       11.3

 HEALTHCARE_INFECTIONS
score        1.2
footnote    53.7

 HCAHPS_PATIENT_SURVEY
patient_survey_star_rating_footnote      95.6
hcahps_answer_percent_footnote           73.6
number_of_completed_surveys_footnote     64.8
survey_response_rate_percent_footnote    64.8

 UNPLANNED_VISITS
footnote    51.5

 TIMELY_EFFECTIVE_CARE
sample       3.4
footnote    33.1

 PAYMENT_AND_VA

In [4]:
cleaned = {}

for name, df in datasets.items():
    # drop footnote columns
    footnote_cols = [col for col in df.columns if "footnote" in col.lower()]
    df = df.drop(columns=footnote_cols)
    
    if "address_line_2" in df.columns:
        df = df.drop(columns=["address_line_2"])
    
    cleaned[name] = df
    print(f"{name}: dropped {len(footnote_cols)} footnote cols → now {df.shape[1]} columns")

hospital_general_info: dropped 6 footnote cols → now 32 columns
readmissions_and_deaths: dropped 1 footnote cols → now 17 columns
medicare_spending: dropped 0 footnote cols → now 13 columns
complications_and_hais: dropped 0 footnote cols → now 11 columns
healthcare_infections: dropped 1 footnote cols → now 14 columns
hcahps_patient_survey: dropped 4 footnote cols → now 18 columns
unplanned_visits: dropped 1 footnote cols → now 19 columns
timely_effective_care: dropped 1 footnote cols → now 15 columns
payment_and_value: dropped 2 footnote cols → now 20 columns


In [5]:
for name, df in cleaned.items():
    id_cols = [col for col in df.columns if any(
        keyword in col.lower() for keyword in ["facility_id", "provider", "ccn"]
    )]
    print(f"{name}: {id_cols}")

hospital_general_info: ['facility_id']
readmissions_and_deaths: ['facility_id']
medicare_spending: ['facility_id']
complications_and_hais: ['cms_certification_number_ccn']
healthcare_infections: ['facility_id']
hcahps_patient_survey: ['facility_id']
unplanned_visits: ['facility_id']
timely_effective_care: ['facility_id']
payment_and_value: ['facility_id']


In [7]:
cleaned["complications_and_hais"] = cleaned["complications_and_hais"].rename(
    columns={"cms_certification_number_ccn": "facility_id"}
)

for name, df in cleaned.items():
    has_id = "facility_id" in df.columns
    print(f"{name}: facility_id " if has_id else f"{name}: NO facility_id!")

hospital_general_info: facility_id 
readmissions_and_deaths: facility_id 
medicare_spending: facility_id 
complications_and_hais: facility_id 
healthcare_infections: facility_id 
hcahps_patient_survey: facility_id 
unplanned_visits: facility_id 
timely_effective_care: facility_id 
payment_and_value: facility_id 


In [9]:
import os

clean_path = "../Data/cleaned/"
os.makedirs(clean_path, exist_ok=True)

for name, df in cleaned.items():
    filepath = clean_path + f"{name}_cleaned.csv"
    df.to_csv(filepath, index=False)
    print(f"saved {name}_cleaned.csv")

print("\nall cleaned datasets saved!")

saved hospital_general_info_cleaned.csv
saved readmissions_and_deaths_cleaned.csv
saved medicare_spending_cleaned.csv
saved complications_and_hais_cleaned.csv
saved healthcare_infections_cleaned.csv
saved hcahps_patient_survey_cleaned.csv
saved unplanned_visits_cleaned.csv
saved timely_effective_care_cleaned.csv
saved payment_and_value_cleaned.csv

all cleaned datasets saved!
